Zinhal
======

### Import

In [1]:
import os
import numpy as np
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import chess
from chess import pgn
from tqdm import tqdm

## Data preprocessing

### Loading data

In [2]:
def get_number_of_games(file_path):
    number_of_games = 0
    with open(file_path, 'r') as pgn_file:
        while True:
            if not pgn.skip_game(pgn_file):
                break
            number_of_games += 1
    return number_of_games
    

def load_pgn(file_path, offset):
    games = np.memmap(filename="../lib/data/npy/games.npy", dtype='object', mode="r+")
    with open(file_path, 'r') as pgn_file:
        i = offset
        while True:
            game = pgn.read_game(pgn_file)
            if game is None:
                break
            games[i] = game
            i += 1
    del games
    return i

files = [file for file in os.listdir("../lib/data/pgn") if file.endswith(".pgn")]
LIMIT_OF_FILES = min(len(files), 30)
number_of_games = 0
for file in tqdm(files[:LIMIT_OF_FILES]):
    number_of_games += get_number_of_games(f"../lib/data/pgn/{file}")

games = np.memmap(filename="../lib/data/npy/games.npy", dtype='object', mode="w+", shape=(number_of_games))
del games
offset = 0
for file in tqdm(files[:LIMIT_OF_FILES]):
    offset = load_pgn(f"../lib/data/pgn/{file}", offset)
games = np.memmap(filename="../lib/data/npy/games.npy", dtype='object', mode="r+")

100%|██████████| 30/30 [03:17<00:00,  6.59s/it]


In [3]:
print(f"Games parsed: {len(games)}")

Games parsed: 247364


### Convert data into tensors

In [4]:
from ridoc import generate_eval_lables

In [5]:
positions, results = generate_eval_lables(games)
print(f"Number of samples: {len(results)}")

100%|██████████| 247364/247364 [31:29<00:00, 130.88it/s]

Number of samples: 19796735


## Preliminary actions

In [6]:
from jesinia import EvalDataset
from violet import EvalModel

In [ ]:
dataset = EvalDataset(positions, results)

dataloader = DataLoader(dataset, batch_size=64)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = EvalModel().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

Using device: cuda


## Traning

In [8]:
num_epochs = 30
for epoch in range(num_epochs):
    start_time = time.time()
    model.train()
    running_loss = 0.0
    for inputs, labels in tqdm(dataloader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs, labels)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        running_loss += loss.item()
    end_time = time.time()
    epoch_time = end_time - start_time
    minutes: int = int(epoch_time // 60)
    seconds: int = int(epoch_time) - minutes * 60
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {running_loss / len(dataloader):.4f}, Time: {minutes}m{seconds}s")

100%|██████████| 309324/309324 [19:57<00:00, 258.29it/s]


Epoch 1/30, Loss: 0.9579, Time: 19m57s


100%|██████████| 309324/309324 [19:58<00:00, 258.11it/s]


Epoch 2/30, Loss: 0.9489, Time: 19m58s


100%|██████████| 309324/309324 [19:58<00:00, 258.03it/s]


Epoch 3/30, Loss: 0.9443, Time: 19m58s


100%|██████████| 309324/309324 [19:58<00:00, 258.14it/s]


Epoch 4/30, Loss: 0.9414, Time: 19m58s


 84%|████████▎ | 258327/309324 [16:40<03:17, 258.12it/s]


KeyboardInterrupt: 

### Save the model

In [ ]:
model_name = f"0_1_{LIMIT_OF_FILES}_{num_epochs}"
torch.save(model.state_dict(), f"../models/e{model_name}.pth")